#  Zermelo’s navigation problem

Problem: Boat navigates from A to B. Boat speed = 1.0. Current speed = 0.5 (half boat speed).

Goal: Reach destination in minimum time (maximize negative time = minimize steps).

In [39]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Environment Creation

In [40]:
# Creating the environment for the movement of the boat in the optimally controlled environment of water
class zermeloenv:
    def __init__(self, grid_size = 20, dt = 0.05):
        # Initiate the grid size to be 20x20
        self.grid_size = grid_size
        # Initiate the time step per action, taken small value to ensure smoother motion
        self.dt = dt
        # Initiate the speed of the boat
        self.boat_speed = 1.0
        # Inititiate the speed of the current
        self.current_speed = np.array([0.5, 0.0])

        # Start Coordinates: (0,0)
        self.start = np.array([0.0, 0.0])
        # Goal Coordinates: (grid_size-1, grid_size-1) i.e. (19, 19)
        self.goal_grid = np.array([grid_size - 1, grid_size - 1])

        # 8 discrete heading directions (in radians) which shows North, North-East, East, South-East, South, South-West, West, North-West
        self.n_actions = 8
        # Creates angles 0, pi/4, pi/2, 3p/4, pi, 5pi/4, 3pi/2, 7pi/4
        self.headings = [2 * np.pi * i / self.n_actions for i in range(self.n_actions)]

        # Maximum number of steps allowed (This is the episode timeout to prevent infinite wandering)
        self.max_steps = 500
        # Initialize the environment
        self._reset_state()

    def _continuous_to_grid(self, pos):
        # Map continuous [0,1]x[0,1] position to grid cell and converts continuous position to grid index

        # Convert x coordinate to integer cell index in a continuous space [0,1]
        gx = int(np.clip(pos[0] * (self.grid_size - 1), 0, self.grid_size - 1))
        
        # Convert y coordinate to integer cell index in a continuous space [0,1]
        gy = int(np.clip(pos[1] * (self.grid_size - 1), 0, self.grid_size - 1))
        return gx, gy

    def _reset_state(self):
        # Internal Reset

        # Reset the position using after copying to avoid modifying the original array
        self.pos = self.start.copy()
        # Reset step counter
        self.steps = 0
        # Return grid state
        return self._continuous_to_grid(self.pos)

    def reset(self):
        # Public Rest (Can be used for external use)
        return self._reset_state()

    def step(self, action):
        # Define the step taken by the boat in the newly made grid environment

        # Map the action index to angle which we get in the form of 8 discrete heading directions (in radians) which shows North, North-East, East, South-East, South, South-West, West, North-West
        theta = self.headings[action]

        # Boat Velocity Vector is obtained after scaling boat speed with the Unit Direction Vector
        boat_vel = np.array([np.cos(theta), np.sin(theta)]) * self.boat_speed

        # Total Velocity is calculated using formula suitable for Zermelo Navigation Setting
        total_vel = boat_vel + self.current_speed

        # Update the position of the boat
        self.pos = self.pos + total_vel * self.dt

        # Define the boundaries such that the boat cannot leave the grid and clip to world [0,1]x[0,1]
        self.pos = np.clip(self.pos, 0.0, 1.0)

        # Increment step count on each step
        self.steps += 1

        # Convet the position in continuous space to grid state
        gx, gy = self._continuous_to_grid(self.pos)

        # Time Penalty of each step 
        done = False
        reward = -1
        
        # Check for goal and assign reward for it
        if gx == self.goal_grid[0] and gy == self.goal_grid[1]:
            reward = 100
            done = True

        # Check timeout case and assign reward for it => Unnecessary wandering is penalised
        elif self.steps >= self.max_steps:
            reward = -50
            done = True
        
        # Return the transition state
        return (gx, gy), reward, done

    def render_path(self, path, title="Zermelo Path"):
        # Define the path for Zermelo path in a plot

        # Convert Grid to continuous
        fig, ax = plt.subplots(figsize=(6, 6))
        xs = [p[0] / (self.grid_size - 1) for p in path]
        ys = [p[1] / (self.grid_size - 1) for p in path]
        
        # Draw current arrows
        ax.quiver(np.zeros(10), np.linspace(0, 1, 10),
                  np.ones(10) * 0.5, np.zeros(10),
                  alpha=0.3, color='cyan', scale=5, label='Current')
        
        # Get the path trajectory
        ax.plot(xs, ys, 'b-o', markersize=3, label='Path')
        
        # Get the Start
        ax.plot(xs[0], ys[0], 'go', markersize=10, label='Start')
        
        # Get the goal
        ax.plot(xs[-1], ys[-1], 'r*', markersize=15, label='Goal')
        
        # Define the labels and plot the Path
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("x"); ax.set_ylabel("y")
        ax.set_title(title); ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        # Save the path in q1_path.png
        plt.savefig("q1_path.png", dpi=100)
        plt.close()

# Q-Learning Agent

In [41]:
class QLearningAgent:
    
    def __init__(self, grid_size, n_actions, alpha=0.1, gamma=0.99, epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995):
        
        # Construct the Q-table, (s,a) i.e. (x, y, a)
        self.Q = np.zeros((grid_size, grid_size, n_actions))
        # Define Alpha value i.e.  Learning Rate
        self.alpha = alpha

        # Define the Gamma value i.e. Discount Factor
        self.gamma = gamma

        # Define the epsilon values 
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Define the actions
        self.n_actions = n_actions

    def choose_action(self, state):
        # Select the action to update the Q-value

        # Exploration for < epsilon
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        
        # Otherwise exploitation
        gx, gy = state
        # Get the max q value for Q-learning
        return int(np.argmax(self.Q[gx, gy]))

    def update(self, s, a, r, s_next, done):
        # Update the Q-value
        gx, gy = s
        nx, ny = s_next
        # Calculate the TD Target with argmax for Q-learning
        target = r + (0 if done else self.gamma * np.max(self.Q[nx, ny]))
        # Update the Q-learning value by alpha times TD Error
        self.Q[gx, gy, a] += self.alpha * (target - self.Q[gx, gy, a])

    def decay_epsilon(self):
        # Exponential Decay in epsilon until the minimum threshold i.e. epsilon = 0.05
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

# Q-Learning Training Loop

In [42]:
def train(n_episodes):

    # Initialize the environment
    env = zermeloenv(grid_size=20)

    # Initialize the Agent
    agent = QLearningAgent(grid_size=20, n_actions=env.n_actions)

    rewards_per_ep = []
    steps_per_ep = []
    solved_eps = []

    # Loop over all episodes
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0

        # Loop over each step
        for _ in range(env.max_steps):
            # Chose the action for a given step
            action = agent.choose_action(state)
            
            # Define the step for a given action
            next_state, reward, done = env.step(action)
            
            # Update Q based on the State, Action, Reward and Next State
            agent.update(state, action, reward, next_state, done)
            
            # Update the state to the next state we get after taking the step 
            state = next_state

            # Accumulate reward
            total_reward += reward
            if done:
                break

        # After each episode, decay the epsilon based on the above define decay
        agent.decay_epsilon()

        # Store the rewards per episode and steps per episode
        rewards_per_ep.append(total_reward)
        steps_per_ep.append(env.steps)
        
        # Check if reward is 100. If yes, episode gives the optimal solution
        if reward == 100:
            solved_eps.append(ep)

    # Evaluate greedy policy
    state = env.reset()
    path = [env._continuous_to_grid(env.pos)]
    for _ in range(env.max_steps):
        gx, gy = state
        action = int(np.argmax(agent.Q[gx, gy]))
        state, reward, done = env.step(action)
        path.append(state)
        if done:
            break

    env.render_path(path, title=f"Zermelo: Optimal Path (steps={len(path)})")

    # Plot training curve
    window = 100
    smoothed = np.convolve(rewards_per_ep, np.ones(window) / window, mode='valid')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(rewards_per_ep, alpha=0.3, color='blue')
    axes[0].plot(range(window - 1, len(rewards_per_ep)), smoothed, color='red', label=f'{window}-ep avg')
    axes[0].set_title("Training Reward per Episode")
    axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Total Reward")
    axes[0].legend()

    # Plot Step Curve
    axes[1].plot(steps_per_ep, alpha=0.3, color='green')
    s2 = np.convolve(steps_per_ep, np.ones(window) / window, mode='valid')
    axes[1].plot(range(window - 1, len(steps_per_ep)), s2, color='orange', label=f'{window}-ep avg')
    axes[1].set_title("Steps to Reach Goal per Episode")
    axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Steps")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"q1_training_{n_episodes}.png", dpi=100)
    plt.close()


    print(f"Q1 Zermelo Navigation:")
    print(f"  Episodes solved: {len(solved_eps)} / {n_episodes}")
    if solved_eps:
        print(f"  First solved at episode: {solved_eps[0]}")
    print(f"  Final greedy path length: {len(path)} steps")
    print(f"  Saved: q1_path.png, q1_training_{n_episodes}.png")
    return agent, rewards_per_ep

# Test on number of episodes

In [43]:
agent, rewards = train(n_episodes=1000)

Q1 Zermelo Navigation:
  Episodes solved: 985 / 1000
  First solved at episode: 1
  Final greedy path length: 32 steps
  Saved: q1_path.png, q1_training_1000.png


In [44]:
agent, rewards = train(n_episodes=2000)

Q1 Zermelo Navigation:
  Episodes solved: 1979 / 2000
  First solved at episode: 0
  Final greedy path length: 30 steps
  Saved: q1_path.png, q1_training_2000.png


In [45]:
agent, rewards = train(n_episodes=3000)

Q1 Zermelo Navigation:
  Episodes solved: 2978 / 3000
  First solved at episode: 1
  Final greedy path length: 33 steps
  Saved: q1_path.png, q1_training_3000.png


In [46]:
agent, rewards = train(n_episodes=5000)

Q1 Zermelo Navigation:
  Episodes solved: 4980 / 5000
  First solved at episode: 4
  Final greedy path length: 32 steps
  Saved: q1_path.png, q1_training_5000.png


In [47]:
agent, rewards = train(n_episodes=10000)

Q1 Zermelo Navigation:
  Episodes solved: 9975 / 10000
  First solved at episode: 6
  Final greedy path length: 29 steps
  Saved: q1_path.png, q1_training_10000.png


In [48]:
agent, rewards = train(n_episodes=50000)

Q1 Zermelo Navigation:
  Episodes solved: 49988 / 50000
  First solved at episode: 0
  Final greedy path length: 28 steps
  Saved: q1_path.png, q1_training_50000.png


In [49]:
agent, rewards = train(n_episodes=100000)

Q1 Zermelo Navigation:
  Episodes solved: 99986 / 100000
  First solved at episode: 0
  Final greedy path length: 29 steps
  Saved: q1_path.png, q1_training_100000.png


In [50]:
agent, rewards = train(n_episodes=1000000)

Q1 Zermelo Navigation:
  Episodes solved: 999981 / 1000000
  First solved at episode: 0
  Final greedy path length: 27 steps
  Saved: q1_path.png, q1_training_1000000.png
